In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- Shared Greater Manila grid definition ---
LAT_MIN = 14.00
LON_MIN = 120.50
CELL_SIZE = 0.009  # approximately 1 km

N_ROWS = 145
N_COLS = 123

DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'greater_manila_grid_water_fraction'

rows = ee.List.sequence(0, N_ROWS - 1)
cols = ee.List.sequence(0, N_COLS - 1)


def make_cell(row, col):
    row = ee.Number(row)
    col = ee.Number(col)

    lat = ee.Number(LAT_MIN).add(row.multiply(CELL_SIZE))
    lon = ee.Number(LON_MIN).add(col.multiply(CELL_SIZE))

    geometry = ee.Geometry.Rectangle(
        [lon, lat, lon.add(CELL_SIZE), lat.add(CELL_SIZE)],
        proj='EPSG:4326',
        geodesic=False
    )

    cell_id = (
        row.format('%03d')
        .cat('_')
        .cat(col.format('%03d'))
    )

    return ee.Feature(geometry).set({
        'cell_id': cell_id,
        'row': row,
        'col': col,
        'longitude': lon.add(CELL_SIZE / 2),
        'latitude': lat.add(CELL_SIZE / 2)
    })


def make_row(row):
    return cols.map(lambda col: make_cell(row, col))


# 145 rows × 123 columns = 17,835 cells.
focus_grid = ee.FeatureCollection(rows.map(make_row).flatten())

print('Expected grid cells:', N_ROWS * N_COLS)
print('Created grid cells:', focus_grid.size().getInfo())


# --- Water fraction ---
# `unmask(0)` ensures places with no detected water are counted as land.
water_over_50 = (
    ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
    .select('occurrence')
    .unmask(0)
    .gt(50)
    .rename('water_fraction_pixel')
)

water_grid = water_over_50.reduceRegions(
    collection=focus_grid,
    reducer=ee.Reducer.mean(),
    scale=30,
    tileScale=4
)


def format_water_cell(feature):
    water_frac = ee.Number(feature.get('mean'))

    return ee.Feature(feature.geometry()).set({
        'cell_id': feature.get('cell_id'),
        'row': feature.get('row'),
        'col': feature.get('col'),
        'longitude': feature.get('longitude'),
        'latitude': feature.get('latitude'),
        'water_frac': water_frac,
        'is_water': water_frac.gt(0.5)
    })


grid_with_water = water_grid.map(format_water_cell)

# Export the table. Keep water cells: they will be excluded later in analysis,
# but must stay flagged in the table.
task = ee.batch.Export.table.toDrive(
    collection=grid_with_water.select([
        'cell_id',
        'row',
        'col',
        'longitude',
        'latitude',
        'water_frac',
        'is_water'
    ]),
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Greater Manila grid and water-fraction export started')
print('Task ID:', task.id)
print('Expected cells: 17,835')

Expected grid cells: 17835
Created grid cells: 17835
✅ Greater Manila grid and water-fraction export started
Task ID: TMYO5FO2UI6WOVFLMMFQE3KI
Expected cells: 17,835


In [ ]:
print(task.status())

{'state': 'COMPLETED', 'description': 'greater_manila_grid_water_fraction', 'priority': 100, 'creation_timestamp_ms': 1794719327771, 'update_timestamp_ms': 1794719909070, 'start_timestamp_ms': 1794719330643, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1-bU5d-UWd39u4OkyDb09gBu3DzCrayi9'], 'attempt': 1, 'batch_eecu_usage_seconds': 763.3495483398438, 'id': 'TMYO5FO2UI6WOVFLMMFQE3KI', 'name': 'projects/avilight-483312/operations/TMYO5FO2UI6WOVFLMMFQE3KI'}


In [ ]:
import pandas as pd

grid = pd.read_csv('greater_manila_grid_water_fraction.csv')

print('Total cells:', len(grid))
print('Expected cells:', 145 * 123)

print('\nUP Diliman cross-check:')
print(grid.loc[
    grid['cell_id'] == '072_062',
    ['cell_id', 'row', 'col', 'latitude', 'longitude', 'water_frac', 'is_water']
])

print('\nWater-heavy cells:', grid['is_water'].sum())

Total cells: 17835
Expected cells: 17835

UP Diliman cross-check:
      cell_id   row   col  latitude  longitude  water_frac  is_water
8918  072_062  72.0  62.0   14.6525   121.0625         0.0         0

Water-heavy cells: 4556


In [ ]:
grid.to_parquet('grid_water_fraction.parquet', index=False)

In [ ]:
import ee

# This code assumes `grid_with_water` already exists in this notebook.
# It must contain: cell_id, row, col, water_frac, is_water, and cell geometry.

DRIVE_FOLDER = 'Satellite Datasets'
START_YEAR = 2014
END_YEAR = 2025


def mask_scale_ndvi(image):
    """Keep quality-approved MODIS NDVI and apply its scale factor."""
    qa_mask = image.select('SummaryQA').lte(1)

    return (
        image.select('NDVI')
        .updateMask(qa_mask)
        .multiply(0.0001)
        .rename('ndvi')
    )


def monthly_ndvi_collection(year):
    """Return 12 monthly NDVI images for one calendar year."""
    year_start = ee.Date.fromYMD(year, 1, 1)
    month_indices = ee.List.sequence(0, 11)

    def make_month(month_index):
        d0 = year_start.advance(ee.Number(month_index), 'month')
        d1 = d0.advance(1, 'month')

        monthly_images = (
            ee.ImageCollection('MODIS/061/MOD13A1')
            .filterDate(d0, d1)
            .map(mask_scale_ndvi)
        )

        ndvi = monthly_images.mean().rename('ndvi')
        ndvi_valid = ndvi.mask().rename('ndvi_valid')

        return (
            ndvi.addBands(ndvi_valid)
            .set({
                'year': d0.get('year'),
                'month': d0.get('month')
            })
        )

    return ee.ImageCollection.fromImages(
        month_indices.map(make_month)
    )


def reduce_month_to_grid(image):
    """Calculate mean NDVI and valid-pixel share for every grid cell."""
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'ndvi': feature.get('ndvi'),
            'ndvi_valid': feature.get('ndvi_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


# Launch one export task per year.
ndvi_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly_ndvi = monthly_ndvi_collection(year)

    yearly_table = yearly_ndvi.map(
        reduce_month_to_grid
    ).flatten()

    export_name = f'grid_ndvi_{year}'

    task = ee.batch.Export.table.toDrive(
        collection=yearly_table,
        description=export_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=export_name,
        fileFormat='CSV'
    )

    task.start()
    ndvi_tasks.append(task)

    print(f'✅ Started {export_name} | Task ID: {task.id}')

print(f'\nStarted {len(ndvi_tasks)} Greater Manila NDVI export tasks.')
print('Each output should contain 17,835 cells × 12 months = 214,020 rows.')

✅ Started grid_ndvi_2014 | Task ID: JZAIH45ARDRCX3WSUKMZIZXR
✅ Started grid_ndvi_2015 | Task ID: DHBEIGXHT6OK6LMB6BRQALZF
✅ Started grid_ndvi_2016 | Task ID: DFQT6MMOGZWR4SNAKCTWXNE4
✅ Started grid_ndvi_2017 | Task ID: J545VQ7VXIECYJVJTAFMMQY4
✅ Started grid_ndvi_2018 | Task ID: OPIHJHPMBSQHHSFB7KP4OAZJ
✅ Started grid_ndvi_2019 | Task ID: ZSJBYFWGMV3SQ5YT3SAIZ3TA
✅ Started grid_ndvi_2020 | Task ID: 2J3HOPCMUVFVYLH7Q4YYIX7H
✅ Started grid_ndvi_2021 | Task ID: RJDDVCI6ZJ3T2TYN2SMHOOPX
✅ Started grid_ndvi_2022 | Task ID: 7ADYIIL4ZPXCCLETADKH2VRF
✅ Started grid_ndvi_2023 | Task ID: 53ICTLAGPI2MJNMRUXXEMP4C
✅ Started grid_ndvi_2024 | Task ID: KV3M37XYSFFFMNZLXDMQPO4D
✅ Started grid_ndvi_2025 | Task ID: U4G6PLXNM7CYGADGRYNF32ZJ

Started 12 Greater Manila NDVI export tasks.
Each output should contain 17,835 cells × 12 months = 214,020 rows.


In [ ]:
DRIVE_FOLDER = 'Satellite Datasets'
START_YEAR = 2014
END_YEAR = 2025


def mask_alan(image):
    """Keep high-quality Black Marble night-light observations."""
    quality_mask = image.select('Mandatory_Quality_Flag').lte(1)

    return (
        image.select('DNB_BRDF_Corrected_NTL')
        .updateMask(quality_mask)
        .rename('alan')
    )


def monthly_alan_collection(year):
    """Return 12 monthly-median ALAN images for one year."""
    year_start = ee.Date.fromYMD(year, 1, 1)
    month_indices = ee.List.sequence(0, 11)

    def make_month(month_index):
        d0 = year_start.advance(ee.Number(month_index), 'month')
        d1 = d0.advance(1, 'month')

        monthly_images = (
            ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
            .filterDate(d0, d1)
            .map(mask_alan)
        )

        alan = monthly_images.median().rename('alan')
        alan_valid = alan.mask().rename('alan_valid')

        return (
            alan.addBands(alan_valid)
            .set({
                'year': d0.get('year'),
                'month': d0.get('month')
            })
        )

    return ee.ImageCollection.fromImages(
        month_indices.map(make_month)
    )


def reduce_alan_month_to_grid(image):
    """Calculate monthly ALAN and valid-pixel share for every grid cell."""
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'alan': feature.get('alan'),
            'alan_valid': feature.get('alan_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


alan_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly_alan = monthly_alan_collection(year)

    yearly_table = yearly_alan.map(
        reduce_alan_month_to_grid
    ).flatten()

    export_name = f'grid_alan_{year}'

    task = ee.batch.Export.table.toDrive(
        collection=yearly_table,
        description=export_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=export_name,
        fileFormat='CSV'
    )

    task.start()
    alan_tasks.append(task)

    print(f'✅ Started {export_name} | Task ID: {task.id}')

print(f'\nStarted {len(alan_tasks)} Greater Manila ALAN export tasks.')
print('Each output should contain 17,835 cells × 12 months = 214,020 rows.')

✅ Started grid_alan_2014 | Task ID: 6QDAJWSTNKGSNBNH3UVNKUIR
✅ Started grid_alan_2015 | Task ID: L6CRLFQEIDGDWLKFC6V7DJYM
✅ Started grid_alan_2016 | Task ID: LA6SYPGBCME7UYOWRLIDTRHV
✅ Started grid_alan_2017 | Task ID: UCYUZMPXP57IFSLGCTM6EHNG
✅ Started grid_alan_2018 | Task ID: N5NOVOPUXTD4CL4TADWP4UOL
✅ Started grid_alan_2019 | Task ID: JCBDGQM64NEANN4BA3JQN5YU
✅ Started grid_alan_2020 | Task ID: GMZFKFIOD2PO7L7GVGCQLI7A
✅ Started grid_alan_2021 | Task ID: L6G2KF6UO2DB3UPJZMRNW22B
✅ Started grid_alan_2022 | Task ID: 67GMTE56YJS43X557BG7CQ52
✅ Started grid_alan_2023 | Task ID: J4BPTXAK4N6A6F2ELSSXYJ73
✅ Started grid_alan_2024 | Task ID: 3BTVXCH4QDDC6ZEYZGJXMIXX
✅ Started grid_alan_2025 | Task ID: UMXAGIGJ7QHXTDX7TFZELPNY

Started 12 Greater Manila ALAN export tasks.
Each output should contain 17,835 cells × 12 months = 214,020 rows.


In [ ]:
DRIVE_FOLDER = 'Satellite Datasets'
START_YEAR = 2014
END_YEAR = 2025


def mask_scale_lst(image):
    """Keep approved LST pixels and convert values to degrees Celsius."""
    qa_mask = image.select('QC_Day').bitwiseAnd(3).lte(1)

    return (
        image.select('LST_Day_1km')
        .updateMask(qa_mask)
        .multiply(0.02)
        .subtract(273.15)
        .rename('lst_day')
    )


def monthly_lst_collection(year):
    """Return 12 monthly-mean daytime LST images for one year."""
    year_start = ee.Date.fromYMD(year, 1, 1)
    month_indices = ee.List.sequence(0, 11)

    def make_month(month_index):
        d0 = year_start.advance(ee.Number(month_index), 'month')
        d1 = d0.advance(1, 'month')

        monthly_images = (
            ee.ImageCollection('MODIS/061/MOD11A2')
            .filterDate(d0, d1)
            .map(mask_scale_lst)
        )

        lst_day = monthly_images.mean().rename('lst_day')
        lst_valid = lst_day.mask().rename('lst_valid')

        return (
            lst_day.addBands(lst_valid)
            .set({
                'year': d0.get('year'),
                'month': d0.get('month')
            })
        )

    return ee.ImageCollection.fromImages(
        month_indices.map(make_month)
    )


def reduce_lst_month_to_grid(image):
    """Calculate monthly daytime LST and valid-pixel share by grid cell."""
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'lst_day': feature.get('lst_day'),
            'lst_valid': feature.get('lst_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


lst_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly_lst = monthly_lst_collection(year)

    yearly_table = yearly_lst.map(
        reduce_lst_month_to_grid
    ).flatten()

    export_name = f'grid_lst_day_{year}'

    task = ee.batch.Export.table.toDrive(
        collection=yearly_table,
        description=export_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=export_name,
        fileFormat='CSV'
    )

    task.start()
    lst_tasks.append(task)

    print(f'✅ Started {export_name} | Task ID: {task.id}')

print(f'\nStarted {len(lst_tasks)} Greater Manila LST export tasks.')
print('Each output should contain 17,835 cells × 12 months = 214,020 rows.')

✅ Started grid_lst_day_2014 | Task ID: G5EJXUMLFHARSRDNH5J6W6N3
✅ Started grid_lst_day_2015 | Task ID: XJPW6AD6WXOYGWNRAIYOCDRR
✅ Started grid_lst_day_2016 | Task ID: IFVLC7TG3VOC5PTNGRAR2KZH
✅ Started grid_lst_day_2017 | Task ID: KNJLF66MSVCKJQFYNP4AG3IV
✅ Started grid_lst_day_2018 | Task ID: VAK3XUNYCQ4IWXEPH4C7BKSP
✅ Started grid_lst_day_2019 | Task ID: CEH2ZCTSD2MYZ4GJIWBTX7CV
✅ Started grid_lst_day_2020 | Task ID: POMJLTWT6X2JTPHFLIEUOVQ3
✅ Started grid_lst_day_2021 | Task ID: I5DGPTOXJ5QRRXXUQ4ZJVN5U
✅ Started grid_lst_day_2022 | Task ID: CUQWSQHDMTE6TYBBKON43EDD
✅ Started grid_lst_day_2023 | Task ID: U45JX22IYMCSP6K2PBZDOKZD
✅ Started grid_lst_day_2024 | Task ID: TFV527EEMUNWHH56LX77NJEA
✅ Started grid_lst_day_2025 | Task ID: YKOSLYJAYRHWHGROF2ST62NR

Started 12 Greater Manila LST export tasks.
Each output should contain 17,835 cells × 12 months = 214,020 rows.


In [ ]:
DRIVE_FOLDER = 'Satellite Datasets'
START_YEAR = 2014
END_YEAR = 2025


def monthly_rain_collection(year):
    """Return 12 IMERG monthly-rainfall images in mm/month for one year."""
    year_start = ee.Date.fromYMD(year, 1, 1)
    month_indices = ee.List.sequence(0, 11)

    def make_month(month_index):
        d0 = year_start.advance(ee.Number(month_index), 'month')
        d1 = d0.advance(1, 'month')

        # IMERG V07 precipitation is in mm/hr.
        monthly_images = (
            ee.ImageCollection('NASA/GPM_L3/IMERG_V07')
            .filterDate(d0, d1)
            .select('precipitation')
        )

        hours_in_month = d1.difference(d0, 'day').multiply(24)

        rain_mm = (
            monthly_images
            .mean()
            .multiply(hours_in_month)
            .rename('rain_mm')
        )

        rain_valid = rain_mm.mask().rename('rain_valid')

        return (
            rain_mm.addBands(rain_valid)
            .set({
                'year': d0.get('year'),
                'month': d0.get('month')
            })
        )

    return ee.ImageCollection.fromImages(
        month_indices.map(make_month)
    )


def reduce_rain_month_to_grid(image):
    """Calculate monthly rainfall and valid-pixel share by grid cell."""
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=5000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'rain_mm': feature.get('rain_mm'),
            'rain_valid': feature.get('rain_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


rain_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    yearly_rain = monthly_rain_collection(year)

    yearly_table = yearly_rain.map(
        reduce_rain_month_to_grid
    ).flatten()

    export_name = f'grid_rain_{year}'

    task = ee.batch.Export.table.toDrive(
        collection=yearly_table,
        description=export_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=export_name,
        fileFormat='CSV'
    )

    task.start()
    rain_tasks.append(task)

    print(f'✅ Started {export_name} | Task ID: {task.id}')

print(f'\nStarted {len(rain_tasks)} Greater Manila rainfall export tasks.')
print('Each output should contain 17,835 cells × 12 months = 214,020 rows.')

✅ Started grid_rain_2014 | Task ID: AHMDXMD2A7CIHOA5MHVZP4FD
✅ Started grid_rain_2015 | Task ID: WVP2WIJBFBR74HCKPDOL5BGV
✅ Started grid_rain_2016 | Task ID: FZ2KOERRN7ZCQ4RGT634ZVLF
✅ Started grid_rain_2017 | Task ID: LM6EJAUUA5W223ATSLRQFCXB
✅ Started grid_rain_2018 | Task ID: 7KALLQXRFICPPHD2SNDSEP4T
✅ Started grid_rain_2019 | Task ID: BK7KFVPKASQJPJW5OE7YMBPX
✅ Started grid_rain_2020 | Task ID: V75BXBUCTJTIQ3W2DKZYBNHU
✅ Started grid_rain_2021 | Task ID: WKGQCSHYYXQ5YZVE73ZCRNVJ
✅ Started grid_rain_2022 | Task ID: DXGNZ3FFI5TPSCGNJAG3QBGJ
✅ Started grid_rain_2023 | Task ID: JDG7YMFGL7CZ34DFAIA7RHD6
✅ Started grid_rain_2024 | Task ID: XBQXPVA454WR7QAPL5V7JDQC
✅ Started grid_rain_2025 | Task ID: NFFVALS7OASLH5ZLCYWSLIWD

Started 12 Greater Manila rainfall export tasks.
Each output should contain 17,835 cells × 12 months = 214,020 rows.


In [ ]:
import ee

# Assumes `grid_with_water` already exists in this notebook.

DRIVE_FOLDER = 'Satellite Datasets'

# MCD12Q1 has observed maps through 2024.
# The 2025 analysis row intentionally reuses the 2024 map.
ANALYSIS_YEARS = ee.List.sequence(2014, 2025)

# IGBP LC_Type1 grouping used for this project:
# 0 water
# 1-5 forest
# 6-10 grassland, shrubland, savanna
# 11 wetland
# 12 and 14 cropland / cropland-natural vegetation mosaic
# 13 urban
LC_GROUPS = {
    'forest_frac': [1, 2, 3, 4, 5],
    'grass_shrub_frac': [6, 7, 8, 9, 10],
    'wetland_frac': [11],
    'cropland_frac': [12, 14],
    'urban_frac': [13],
    'water_lc_frac': [0]
}


# Province boundaries for the Trend Explorer.
provinces = (
    ee.FeatureCollection('FAO/GAUL/2025/level2')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Philippines'))
)


def make_binary_group(land_cover, codes, band_name):
    """Return 1 for pixels in the requested IGBP codes; 0 otherwise."""
    return (
        land_cover
        .remap(codes, [1] * len(codes), 0)
        .rename(band_name)
    )


def yearly_landcover_shares(year):
    """
    Make six 0/1 land-cover group bands for one analysis year.
    For 2025, use the latest available MCD12Q1 map (2024).
    """
    year = ee.Number(year)
    lc_year_used = year.min(2024)

    d0 = ee.Date.fromYMD(lc_year_used, 1, 1)
    d1 = d0.advance(1, 'year')

    land_cover = (
        ee.ImageCollection('MODIS/061/MCD12Q1')
        .filterDate(d0, d1)
        .first()
        .select('LC_Type1')
    )

    share_bands = ee.Image.cat([
        make_binary_group(land_cover, LC_GROUPS['urban_frac'], 'urban_frac'),
        make_binary_group(land_cover, LC_GROUPS['forest_frac'], 'forest_frac'),
        make_binary_group(land_cover, LC_GROUPS['cropland_frac'], 'cropland_frac'),
        make_binary_group(
            land_cover,
            LC_GROUPS['grass_shrub_frac'],
            'grass_shrub_frac'
        ),
        make_binary_group(land_cover, LC_GROUPS['wetland_frac'], 'wetland_frac'),
        make_binary_group(land_cover, LC_GROUPS['water_lc_frac'], 'water_lc_frac')
    ])

    return share_bands.set({
        'year': year,
        'lc_year_used': lc_year_used
    })


landcover_yearly = ee.ImageCollection.fromImages(
    ANALYSIS_YEARS.map(yearly_landcover_shares)
)


def reduce_landcover_to_provinces(image):
    """Compute the share of each land-cover group in every province."""
    reduced = image.reduceRegions(
        collection=provinces,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE'),
            'year': image.get('year'),
            'lc_year_used': image.get('lc_year_used'),
            'urban_frac': feature.get('urban_frac'),
            'forest_frac': feature.get('forest_frac'),
            'cropland_frac': feature.get('cropland_frac'),
            'grass_shrub_frac': feature.get('grass_shrub_frac'),
            'wetland_frac': feature.get('wetland_frac'),
            'water_lc_frac': feature.get('water_lc_frac')
        })

    return reduced.map(format_row)


def reduce_landcover_to_grid(image):
    """Compute land-cover shares in every Greater Manila grid cell."""
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'lc_year_used': image.get('lc_year_used'),
            'urban_frac': feature.get('urban_frac'),
            'forest_frac': feature.get('forest_frac'),
            'cropland_frac': feature.get('cropland_frac'),
            'grass_shrub_frac': feature.get('grass_shrub_frac'),
            'wetland_frac': feature.get('wetland_frac'),
            'water_lc_frac': feature.get('water_lc_frac'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


# Build export tables.
province_landcover = landcover_yearly.map(
    reduce_landcover_to_provinces
).flatten()

grid_landcover = landcover_yearly.map(
    reduce_landcover_to_grid
).flatten()


# Export province land-cover shares.
province_task = ee.batch.Export.table.toDrive(
    collection=province_landcover,
    description='province_landcover_2014_2025',
    folder=DRIVE_FOLDER,
    fileNamePrefix='province_landcover_2014_2025',
    fileFormat='CSV'
)

# Export Greater Manila grid land-cover shares.
grid_task = ee.batch.Export.table.toDrive(
    collection=grid_landcover,
    description='grid_landcover_2014_2025',
    folder=DRIVE_FOLDER,
    fileNamePrefix='grid_landcover_2014_2025',
    fileFormat='CSV'
)

province_task.start()
grid_task.start()

print('✅ Province land-cover export started')
print('Task ID:', province_task.id)

print('✅ Greater Manila grid land-cover export started')
print('Task ID:', grid_task.id)

print('\nExpected province rows: 85 provinces × 12 years = 1,020')
print('Expected grid rows: 17,835 cells × 12 years = 214,020')
print('Note: 2025 rows use the 2024 MCD12Q1 map and have lc_year_used = 2024.')

✅ Province land-cover export started
Task ID: 6JZRUDT4FZAVQIFNCDCZJ6UB
✅ Greater Manila grid land-cover export started
Task ID: VPLH5PJ7SCA4PNW75XSMIULC

Expected province rows: 85 provinces × 12 years = 1,020
Expected grid rows: 17,835 cells × 12 years = 214,020
Note: 2025 rows use the 2024 MCD12Q1 map and have lc_year_used = 2024.


In [ ]:
DRIVE_FOLDER = 'Satellite Datasets'

# Simplified province boundaries for the app.
# 1,000 m tolerance keeps the GeoJSON smaller for map animation.
def format_province_geojson(feature):
    return ee.Feature(
        feature.geometry().simplify(1000),
        {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE')
        }
    )

provinces_geojson = provinces.map(format_province_geojson)


# Square Greater Manila grid boundaries for the Birds & Light map.
def format_grid_geojson(feature):
    return ee.Feature(
        feature.geometry(),
        {
            'cell_id': feature.get('cell_id'),
            'row': feature.get('row'),
            'col': feature.get('col'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        }
    )

grid_geojson = grid_with_water.map(format_grid_geojson)


province_geo_task = ee.batch.Export.table.toDrive(
    collection=provinces_geojson,
    description='provinces_geojson',
    folder=DRIVE_FOLDER,
    fileNamePrefix='provinces',
    fileFormat='GeoJSON'
)

grid_geo_task = ee.batch.Export.table.toDrive(
    collection=grid_geojson,
    description='greater_manila_grid_geojson',
    folder=DRIVE_FOLDER,
    fileNamePrefix='grid',
    fileFormat='GeoJSON'
)

province_geo_task.start()
grid_geo_task.start()

print('✅ Province GeoJSON export started')
print('Task ID:', province_geo_task.id)

print('✅ Greater Manila grid GeoJSON export started')
print('Task ID:', grid_geo_task.id)

✅ Province GeoJSON export started
Task ID: V2OG5TJNKMZXVCVEUQ7UY7DD
✅ Greater Manila grid GeoJSON export started
Task ID: XUQXQHKC7V7FPP57PUMROCRO


In [ ]:
import pandas as pd
import ee

tracked_prefixes = (
    'grid_',
    'province_',
    'provinces_geojson',
    'greater_manila_grid_geojson'
)

rows = []

for earth_engine_task in ee.batch.Task.list():
    status = earth_engine_task.status()
    description = status.get('description', '')

    if description.startswith(tracked_prefixes):
        rows.append({
            'description': description,
            'state': status.get('state'),
            'error': status.get('error_message', '')
        })

task_status = (
    pd.DataFrame(rows)
    .sort_values(['state', 'description'])
    .reset_index(drop=True)
)

print(task_status[['description', 'state']].to_string(index=False))

failed = task_status[task_status['state'] == 'FAILED']

if len(failed) > 0:
    print('\n❌ Failed tasks:')
    print(failed.to_string(index=False))
else:
    print('\n✅ No failed tasks currently.')

                       description     state
       greater_manila_grid_geojson COMPLETED
                    grid_alan_2014 COMPLETED
                    grid_alan_2015 COMPLETED
                    grid_alan_2016 COMPLETED
                    grid_alan_2017 COMPLETED
                    grid_alan_2018 COMPLETED
                    grid_alan_2019 COMPLETED
                    grid_alan_2020 COMPLETED
                    grid_alan_2021 COMPLETED
                    grid_alan_2022 COMPLETED
                    grid_alan_2023 COMPLETED
                    grid_alan_2024 COMPLETED
                    grid_alan_2025 COMPLETED
          grid_landcover_2014_2025 COMPLETED
                 grid_lst_day_2014 COMPLETED
                 grid_lst_day_2015 COMPLETED
                 grid_lst_day_2016 COMPLETED
                 grid_lst_day_2017 COMPLETED
                 grid_lst_day_2018 COMPLETED
                 grid_lst_day_2019 COMPLETED
                 grid_lst_day_2020 COMPLETED
          

In [ ]:
test_image = monthly_ndvi_collection(2014).first()

up_cell = grid_with_water.filter(
    ee.Filter.eq('cell_id', '072_062')
)

test_feature = (
    test_image
    .reduceRegions(
        collection=up_cell,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )
    .first()
)

print(test_feature.toDictionary().getInfo())

{'cell_id': '072_062', 'col': 62, 'is_water': 0, 'latitude': 14.6525, 'longitude': 121.0625, 'ndvi': 0.5064539589442815, 'ndvi_valid': 1, 'row': 72, 'water_frac': 0}


In [22]:
DRIVE_FOLDER = 'Satellite Datasets'
START_YEAR = 2014
END_YEAR = 2025
MISSING_VALUE = -9999


def reduce_ndvi_v2(image):
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4
    )

    def format_row(feature):
        ndvi_value = ee.Algorithms.If(
            ee.Number(feature.get('ndvi_valid')).eq(0),
            MISSING_VALUE,
            feature.get('ndvi')
        )

        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'ndvi': ndvi_value,
            'ndvi_valid': feature.get('ndvi_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


def reduce_lst_v2(image):
    reduced = image.reduceRegions(
        collection=grid_with_water,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    def format_row(feature):
        lst_value = ee.Algorithms.If(
            ee.Number(feature.get('lst_valid')).eq(0),
            MISSING_VALUE,
            feature.get('lst_day')
        )

        return ee.Feature(None, {
            'cell_id': feature.get('cell_id'),
            'year': image.get('year'),
            'month': image.get('month'),
            'lst_day': lst_value,
            'lst_valid': feature.get('lst_valid'),
            'water_frac': feature.get('water_frac'),
            'is_water': feature.get('is_water')
        })

    return reduced.map(format_row)


# These functions were defined in the previous NDVI/LST replacement cell:
# fixed_monthly_ndvi_collection(year)
# fixed_monthly_lst_collection(year)

v2_tasks = []

for year in range(START_YEAR, END_YEAR + 1):
    ndvi_table = fixed_monthly_ndvi_collection(year).map(
        reduce_ndvi_v2
    ).flatten()

    ndvi_name = f'grid_ndvi_v2_{year}'

    ndvi_task = ee.batch.Export.table.toDrive(
        collection=ndvi_table,
        description=ndvi_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=ndvi_name,
        fileFormat='CSV'
    )

    ndvi_task.start()
    v2_tasks.append(ndvi_task)
    print(f'✅ Started {ndvi_name}')

    lst_table = fixed_monthly_lst_collection(year).map(
        reduce_lst_v2
    ).flatten()

    lst_name = f'grid_lst_day_v2_{year}'

    lst_task = ee.batch.Export.table.toDrive(
        collection=lst_table,
        description=lst_name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=lst_name,
        fileFormat='CSV'
    )

    lst_task.start()
    v2_tasks.append(lst_task)
    print(f'✅ Started {lst_name}')

print(f'\nStarted {len(v2_tasks)} final NDVI/LST replacement tasks.')

✅ Started grid_ndvi_v2_2014
✅ Started grid_lst_day_v2_2014
✅ Started grid_ndvi_v2_2015
✅ Started grid_lst_day_v2_2015
✅ Started grid_ndvi_v2_2016
✅ Started grid_lst_day_v2_2016
✅ Started grid_ndvi_v2_2017
✅ Started grid_lst_day_v2_2017
✅ Started grid_ndvi_v2_2018
✅ Started grid_lst_day_v2_2018
✅ Started grid_ndvi_v2_2019
✅ Started grid_lst_day_v2_2019
✅ Started grid_ndvi_v2_2020
✅ Started grid_lst_day_v2_2020
✅ Started grid_ndvi_v2_2021
✅ Started grid_lst_day_v2_2021
✅ Started grid_ndvi_v2_2022
✅ Started grid_lst_day_v2_2022
✅ Started grid_ndvi_v2_2023
✅ Started grid_lst_day_v2_2023
✅ Started grid_ndvi_v2_2024
✅ Started grid_lst_day_v2_2024
✅ Started grid_ndvi_v2_2025
✅ Started grid_lst_day_v2_2025

Started 24 final NDVI/LST replacement tasks.
